# This is an example how to recreate the LongEval-Sci dataset

In [1]:
DB = "sqlite:///longeval.sqlite3"

In [2]:
from src.database import create_database_engine, create_schema

engine = create_database_engine(DB)
create_schema(engine)

## 1. Log Ingestion
```bash
uv run longeval-cli import-queries data/raw/logs --parser parser.py --database-url $DB
```

In [3]:
from pathlib import Path

from src.importing import ImportConfig, import_clicks, import_queries
from src.importing.loader import load_parser

DB = "sqlite:///longeval.sqlite3"

config = ImportConfig(
    inputs=[Path("data/raw/logs")],
    database_url=DB,
    parser=load_parser("parser.py"),   # the file defines a top-level `parser` object
    batch_size=5000, 
    # strict=False, 
    # max_errors=None,
    limit=5_000_000
)

In [4]:
print(import_queries(config))   # phase 1: queries and rankings
print(import_clicks(config))    # phase 2: clicks; needs phase 1 first

Counter({'searches_seen': 5000000, 'known_mappings': 4693923, 'conflicts': 284550, 'rejected_records': 13233})
Counter({'interactions_seen': 5000000, 'unresolved_searches': 3644071, 'session_mismatches': 957529, 'rejected_records': 530795, 'missing_documents': 458})


## 2. Query Selection

In [5]:
import re

from src.classifying.builtins import (
    ClicksClassifier,
    IsBooleanClassifier,
    WordCountClassifier,
)
from src.classifying.records import Classification, QueryClassifier


class HasDoiClassifier(QueryClassifier):
    name = "has-doi"
    
    doi_regex = re.compile(r"(^|[^0-9])10\.[0-9]{4,9}/[-._;()/:a-zA-Z0-9]+")

    def classify(self, query: str):
        label = "doi" if self.doi_regex.search(query) else "no-doi"
        return Classification(label=label)


class IsStructuredClassifier(QueryClassifier):
    name = "is-structured"
    
    core_fiels = (
        "abstract", "acceptedDate", "arxivId", "authors", "citationCount", "contributors", "createdDate",
        "dataProviders", "depositedDate", "documentType", "doi", "downloadUrl", "fieldOfStudy", "fullText",
        "identifiers", "journals", "links", "magId", "oaiIds", "outputs", "publishedDate", "publisher", "pubmedId",
        "references", "sourceFulltextUrls", "title", "updatedDate", "yearPublished", "language", "_exists_", "id",
    )
    field_lookup = re.compile(
        r"(^|\s|[^a-zA-Z0-9])(" + "|".join(core_fiels) + r"):", re.IGNORECASE
    )

    def classify(self, query: str):
        label = "structured" if self.field_lookup.search(query) else "not-structured"
        return Classification(label=label)


query_classifier = [
    IsBooleanClassifier(),
    WordCountClassifier(),
    HasDoiClassifier(),
    IsStructuredClassifier(),
    ClicksClassifier(
        "clicks", types=["get-pdf", "works"], within_hours=24
    )
]

/Users/jueri/projects/conf26-longeval-sci/code/conf26-longeval-sci/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Apply the classifier

In [6]:
from src.classifying import ClassifierConfig, classify_queries
from src.filtering import load_snapshot_config

# Snapshot name and [from, to) range come from the snapshot's filter YAML.
snapshot_config = load_snapshot_config(Path("data/raw/filters/snapshot-3.yaml"))

config = ClassifierConfig(
    database_url=DB,
    classifier=query_classifier,
    snapshot_config=snapshot_config,
)
print(classify_queries(config))

is-structured: 100%|██████████| 1873088/1873088 [07:41<00:00, 4059.22 queries/s]
clicks: 0 queries [00:00, ? queries/s]

Counter({'classified': 7492351, 'unlabeled': 1})


Inspect the stored verdicts.

In [7]:
import sqlalchemy as sa

from src.database import make_session_factory
from src.models import QueryClassification as QC

# How many queries got which label from which classifier in which scope.
with make_session_factory(engine)() as session:
    rows = session.execute(
        sa.select(QC.snapshot, QC.classifier, QC.label, sa.func.count())
        .group_by(QC.snapshot, QC.classifier, QC.label)
        .order_by(QC.classifier, QC.snapshot, QC.label)
    ).all()

for scope, classifier, label, count in rows:
    print(f"{classifier:14} {scope or '(global)':11} {label:16} {count}")

has-doi        snapshot-3  doi              105232
has-doi        snapshot-3  no-doi           1767856
is-boolean     snapshot-3  0                1423911
is-boolean     snapshot-3  1                449177
is-structured  snapshot-3  not-structured   1469776
is-structured  snapshot-3  structured       403312
word-count     snapshot-3  1                257822
word-count     snapshot-3  10               98250
word-count     snapshot-3  100              9
word-count     snapshot-3  101              43
word-count     snapshot-3  102              8
word-count     snapshot-3  103              10
word-count     snapshot-3  104              11
word-count     snapshot-3  105              15
word-count     snapshot-3  106              36
word-count     snapshot-3  107              43
word-count     snapshot-3  1070             1
word-count     snapshot-3  108              10
word-count     snapshot-3  109              11
word-count     snapshot-3  11               91262
word-count     snapshot-3 

One filter file per snapshot (`filters/snapshot-N.yaml`) keeps the queries that pass the automatic filters, have more than 30 qualifying clicks in that snapshot, and were not removed by manual review. `export queries` as Python: `filter_report` counts what survives each rule, `write_queries_tsv` writes `query_id<TAB>query`.

In [8]:
from src.database import make_session_factory
from src.exporting import write_queries_tsv
from src.filtering import filter_report, load_snapshot_config

with make_session_factory(engine)() as session:
    snapshot_config = load_snapshot_config("data/raw/filters/snapshot-3.yaml")
    report = filter_report(session, snapshot_config)
    output = Path("data/interim/queries-to-check/candidate-queries-snapshot-3.tsv")
    count = write_queries_tsv(session, snapshot_config, output)

    print("snapshot-3:")
    print(f"  {report.total:>8}  all queries")
    for rule, remaining in report.steps:
        print(f"  {remaining:>8}  {rule}")

snapshot-3:
   1873088  all queries
         0  require is-structured in ['0']
         0  require is-boolean in ['0']
         0  require has-doi in ['0']
         0  min clicks >= 5


## Apply manual classification

In [ ]:
from src.classifying.builtins import ManualLabelsClassifier

checked_queries = Path("data/interim/queries-checked/candidate-queries-snapshot-3.tsv")

config = ClassifierConfig(
    database_url=DB,
    classifier=ManualLabelsClassifier(checked_queries),
    snapshot_config=snapshot_config,
)
print(classify_queries(config))

ClassificationStats(considered=54519, classified=1, unlabeled=54518)


In [34]:
with make_session_factory(engine)() as session:
    criteria = load_snapshot_config("data/raw/filters/snapshot-3.yaml")
    report = filter_report(session, criteria)
    output = Path("data/interim/queries-to-check/candidate-queries-snapshot-3.tsv")
    count = write_queries_tsv(session, criteria, output)

    print("snapshot-3:")
    print(f"  {report.total:>8}  all queries")
    for rule, remaining in report.steps:
        print(f"  {remaining:>8}  {rule}")

snapshot-3:
     54519  all queries
     47698  require is-structured in ['not-structured']
     41968  require is-boolean in ['not-boolean']
     41761  require has-doi in ['no-doi']
         1  require manual in ['x']


## 3 Relevance Label Creation

In [ ]:
import optax
from clax import Trainer
from clax.models import DocumentCTRModel
from flax import nnx

from src.clicks.clax_loaders import loaders_from_db
from src.filtering import load_snapshot_config

config = load_snapshot_config("data/raw/filters/snapshot-1.yaml")

# Stream from exported files
# query_doc_ids, train, validation = loaders_from_file("path/to/export", holdout=10, batch_size=1024, max_positions=10)

# Stream data from DB
with make_session_factory(engine)() as session:
    query_doc_ids, train, validation = loaders_from_db(session, config, holdout=10, batch_size=1024, max_positions=10)
    
    # DCTR as a CLAX model
    model = DocumentCTRModel(query_doc_pairs=len(query_doc_ids), rngs=nnx.Rngs(0))
    trainer = Trainer(optimizer=optax.adamw(3e-3), epochs=20)

    # fit
    trainer.train(model, train, validation)

In [ ]:
from src.clicks.scoring import score_from_db, write_scored_docs

# Apply the fitted model to score candidate queries-documents pairs. 
# The scoring function will return a DataFrame with the scores for each candidate pair.
scored = score_from_db(model, query_doc_ids, config, database_url=DB)
write_scored_docs(scored, "data/interim/scored-docs-snapshot-1.parquet")

## 4 Document Collection

## 5 Packaging